In [1]:
import carla 
import math 
import random 
import time 
import os
import numpy as np
import cv2
import matplotlib.pyplot as plt
from map_visualization import MapVisualization
import pprint
import json

import lanelet2
from lanelet2.io import load, Origin
from lanelet2.projection import LocalCartesianProjector, UtmProjector
from lanelet2.core import BasicPoint2d, BasicPoint3d, LineString2d, LaneletMap
from lanelet2.routing import RoutingGraph, RoutingCost, RoutingCostDistance
from lanelet2.traffic_rules import create as create_traffic_rules
from lanelet2.geometry import to2D, project

from util import get_full_trajectory, resample_centerline, compute_s0, slice_centerline, path_length, distance2d

In [2]:
# Connect the client and set up bp library and spawn points
client = carla.Client('localhost', 2000) 
world = client.get_world()
bp_lib = world.get_blueprint_library() 
spawn_points = world.get_map().get_spawn_points() 
tm = client.get_trafficmanager(8000)
tm_port = tm.get_port()

In [ ]:
# Load map and setup
map_path = os.path.expanduser('~/Documents/town_10/backup/lanelet2_map.osm')
assert os.path.exists(map_path), f"Map file not found at {map_path}"
proj = LocalCartesianProjector(Origin(0, 0, 0))
lanelet_map = load(map_path, proj)
traffic_rules = create_traffic_rules("de", "vehicle")
router = RoutingGraph(lanelet_map, traffic_rules, [RoutingCostDistance(1.0)])

# Vehicle position and heading
pt = BasicPoint2d(vehicle.get_location().x, -vehicle.get_location().y)
yaw = vehicle.get_transform().rotation.yaw  # in degrees
yaw_rad = math.radians(yaw)
v_veh = (-math.cos(yaw_rad), -math.sin(yaw_rad))  # Direction in map coordinates

# Find candidate lanelets within 5 meters
candidates = [ll for ll in lanelet_map.laneletLayer if lanelet2.geometry.distance(ll, pt) < 1.0]

# Select lanelet aligned with vehicle's direction
best_lanelet = None
max_dot = -float('inf')
for ll in candidates:
    # 1) Project vehicle point onto this lanelet’s 3D centerline
    pt3d    = BasicPoint3d(pt.x, pt.y, 0.0)
    proj_pt = lanelet2.geometry.project(ll.centerline, pt3d)

    # 2) Find which segment of ll.centerline is closest to proj_pt
    best_i, best_dist = 0, float('inf')
    pts = ll.centerline
    for idx in range(len(pts) - 1):
        p0, p1 = pts[idx], pts[idx+1]
        dx, dy = p1.x - p0.x, p1.y - p0.y
        ux, uy = proj_pt.x - p0.x, proj_pt.y - p0.y
        denom  = dx*dx + dy*dy
        u = (ux*dx + uy*dy) / denom if denom > 0 else 0.0
        u_clamped = max(0.0, min(1.0, u))
        cx, cy = p0.x + u_clamped*dx, p0.y + u_clamped*dy
        d = math.hypot(cx - pt.x, cy - pt.y)
        if d < best_dist:
            best_dist, best_i = d, idx
    
    # 3) Compute this lanelet’s local direction vector
    p0 = pts[best_i]
    p1 = pts[best_i+1]
    v_lane = (p1.x - p0.x, p1.y - p0.y)

    # 4) Compute dot with vehicle’s heading unit vector v_veh
    dot = v_lane[0]*v_veh[0] + v_lane[1]*v_veh[1]
    if dot > max_dot:
        max_dot     = dot
        best_lanelet = ll
        
if best_lanelet is None:
    raise ValueError("No suitable lanelet found near the vehicle")

current_lane = best_lanelet
closest_lanelet_traj = get_full_trajectory(current_lane, router, max_length=50.0, spacing=3.0)
closest_lanelet_traj = np.array(closest_lanelet_traj)

proj_pt = lanelet2.geometry.project(current_lane.centerline, pt3d)
s0 = compute_s0(current_lane.centerline, proj_pt)
print('s0:', s0)
slice_pts = slice_centerline(current_lane.centerline, s0, 50.0)
sliced_trajectory = [(p.x, p.y) for p in slice_pts]
sliced_trajectory_1 = np.array(sliced_trajectory)

following = router.following(current_lane)
branch_lane_1 = following[0]
s0 = compute_s0(branch_lane_1.centerline, proj_pt)
slice_pts = slice_centerline(branch_lane_1.centerline, s0, 50.0)
sliced_trajectory = [(p.x, p.y) for p in slice_pts]
sliced_trajectory_2 = np.array(sliced_trajectory)

following = router.following(current_lane)
branch_lane_2 = following[1]
s0 = compute_s0(branch_lane_2.centerline, proj_pt)
slice_pts = slice_centerline(branch_lane_2.centerline, s0, 50.0)
sliced_trajectory = [(p.x, p.y) for p in slice_pts]
sliced_trajectory_3 = np.array(sliced_trajectory)



all_paths = collect_trajectories(
    lanelet_map,
    router,
    best_lanelet,
    pt3d,
    budget=50.0
)
sliced_trajectory = [(p.x, p.y) for p in all_paths[0]]
sliced_trajectory_4 = np.array(sliced_trajectory)
print(sliced_trajectory_4)

sliced_trajectory = [(p.x, p.y) for p in all_paths[1]]
sliced_trajectory_5 = np.array(sliced_trajectory)
print(sliced_trajectory_5)



In [ ]:
'''
# Get successors of the current lanelet
current_successors = router.following(current_lane)

# Get predecessors of the current lanelet
predecessors = router.previous(current_lane)

# Get successors of the predecessors
predecessor_successors = []
for pred in predecessors:
    predecessor_successors.extend(router.following(pred))

# Combine successors, removing duplicates
all_successors = list(set(current_successors + predecessor_successors))
'''

In [ ]:
def collect_trajectories_v3(lanelet_map, router, current_lane, proj_pt, budget=50.0,
                           visited=None, connection_tol=15.0, original_lane=None, yaw=None,
                           enter_threshold=2.0, angle_thresh_deg=80.0):
    if visited is None:
        visited = set()
    visited.add(current_lane.id)

    # Set the original_lane on the first call
    if original_lane is None:
        original_lane = current_lane

    # Project ego point onto centerline
    proj_pt3d = project(current_lane.centerline, proj_pt)
    s0 = compute_s0(current_lane.centerline, proj_pt3d)
    main_slice = slice_centerline(current_lane.centerline, s0, budget)

    if path_length(main_slice) >= budget:
        return [main_slice]

    # Freeze lanelet choice if ego hasn’t moved far enough
    if s0 < enter_threshold: # path_length(main_slice) < enter_threshold
        lane_for_successors = original_lane
    else:
        lane_for_successors = current_lane

    successors = router.following(lane_for_successors)
    
    if not successors:
        return [main_slice]

    # Set up heading vector from yaw (in radians)
    if yaw is None:
        if len(main_slice) >= 2:
            dx = main_slice[1].x - main_slice[0].x
            dy = main_slice[1].y - main_slice[0].y
            yaw = math.atan2(dy, dx)
        else:
            yaw = 0.0  # Fallback

    angle_thresh = math.radians(angle_thresh_deg)

    # Recursively expand successors
    trajectories = []
    remaining = budget - path_length(main_slice)
    end_pt = main_slice[-1]
    for succ in successors:
        if succ.id in visited:
            continue
        succ_start = succ.centerline[0]
        if distance2d(end_pt, succ_start) > connection_tol:
            continue

        # Filter based on heading alignment
        if len(succ.centerline) >= 2:
            dx = succ.centerline[1].x - succ.centerline[0].x
            dy = succ.centerline[1].y - succ.centerline[0].y
            succ_yaw = math.atan2(dy, dx)
            diff = abs((succ_yaw - yaw + math.pi) % (2 * math.pi) - math.pi)
            if not (diff < math.radians(30) or abs(diff - math.pi/2) < math.radians(30)):
                continue

        sub_visited = visited.copy()
        sub_trajs = collect_trajectories_v3(
            lanelet_map,
            router,
            succ,
            BasicPoint3d(succ_start.x, succ_start.y, getattr(succ_start, 'z', 0.0)),
            budget=remaining,
            visited=sub_visited,
            connection_tol=connection_tol,
            original_lane=original_lane,  # Pass original_lane through recursion
            yaw=yaw,
            enter_threshold=enter_threshold,
            angle_thresh_deg=angle_thresh_deg
        )
        for st in sub_trajs:
            trajectories.append(main_slice + st)

    if not trajectories:
        return [main_slice]
    return trajectories

In [42]:
settings = world.get_settings()
settings.synchronous_mode = True
settings.fixed_delta_seconds = 0.05
world.apply_settings(settings)
client.set_timeout(10.0)

# Load map and setup
map_path = os.path.expanduser('~/Documents/town_10/backup/lanelet2_map.osm')
assert os.path.exists(map_path), f"Map file not found at {map_path}"
proj = LocalCartesianProjector(Origin(0, 0, 0))
lanelet_map = load(map_path, proj)
traffic_rules = create_traffic_rules("de", "vehicle")
router = RoutingGraph(lanelet_map, traffic_rules, [RoutingCostDistance(1.0)])
trajectory_dict = {}


vehicle_bp = bp_lib.find('vehicle.tesla.model3') #112
vehicle = world.spawn_actor(vehicle_bp, spawn_points[136])

tm.ignore_lights_percentage(vehicle, 100)
vehicle.set_autopilot(True, tm_port)
try:

    second = 0
    start_time = time.time()
    prev_collection_time = start_time

    while True:
        world.tick()
#--------------------------------------------------------------------------------------------------------------------
        current_time = time.time()
        if current_time - prev_collection_time >= 0.1:
            print("second:", second)
            # Reset the collection time
            prev_collection_time = current_time

            # Vehicle position and heading
            pt = BasicPoint2d(vehicle.get_location().x, -vehicle.get_location().y)
            yaw = vehicle.get_transform().rotation.yaw  # in degrees
            yaw_rad = math.radians(yaw)
            v_veh = (-math.cos(yaw_rad), -math.sin(yaw_rad))  # Direction in map coordinates

            # Find candidate lanelets within 5 meters
            candidates = [ll for ll in lanelet_map.laneletLayer if lanelet2.geometry.distance(ll, pt) < 1.0]

            # Select lanelet aligned with vehicle's direction
            best_lanelet = None
            max_dot = -float('inf')
            for ll in candidates:
                # 1) Project vehicle point onto this lanelet’s 3D centerline
                pt3d    = BasicPoint3d(pt.x, pt.y, 0.0)
                proj_pt = lanelet2.geometry.project(ll.centerline, pt3d)

                # 2) Find which segment of ll.centerline is closest to proj_pt
                best_i, best_dist = 0, float('inf')
                pts = ll.centerline
                for idx in range(len(pts) - 1):
                    p0, p1 = pts[idx], pts[idx+1]
                    dx, dy = p1.x - p0.x, p1.y - p0.y
                    ux, uy = proj_pt.x - p0.x, proj_pt.y - p0.y
                    denom  = dx*dx + dy*dy
                    u = (ux*dx + uy*dy) / denom if denom > 0 else 0.0
                    u_clamped = max(0.0, min(1.0, u))
                    cx, cy = p0.x + u_clamped*dx, p0.y + u_clamped*dy
                    d = math.hypot(cx - pt.x, cy - pt.y)
                    if d < best_dist:
                        best_dist, best_i = d, idx
                
                # 3) Compute this lanelet’s local direction vector
                p0 = pts[best_i]
                p1 = pts[best_i+1]
                v_lane = (p1.x - p0.x, p1.y - p0.y)

                # 4) Compute dot with vehicle’s heading unit vector v_veh
                dot = v_lane[0]*v_veh[0] + v_lane[1]*v_veh[1]
                if dot > max_dot:
                    max_dot     = dot
                    best_lanelet = ll

            '''
            all_paths = collect_trajectories_v3(
                lanelet_map,
                router,
                best_lanelet,
                pt3d,
                budget=50.0,
                connection_tol=25.0,
                enter_threshold=2.0,
            )

        
            lanelet_trajectory = [(p.x, p.y) for p in best_lanelet.centerline]
            lanelet_trajectory = np.array(lanelet_trajectory)
            fig, ax = plt.subplots(figsize=(30,30))
            ax.scatter(vehicle.get_location().x, -vehicle.get_location().y, c="blue", s=100)
            ax.scatter(road_boundary_coord[:, 0], road_boundary_coord[:, 1], c="gray")
            #ax.scatter(lanelet_trajectory[2::3, 0], lanelet_trajectory[2::3, 1], c="orange")

            trajectory_dict[second] = {}
            trajectory_dict[second]["trajectories"] = []
            trajectory_dict[second]["vehicle_location"] = [vehicle.get_location().x, -vehicle.get_location().y]
            traj_color = ["red", "blue", "green"]
            succ_traj_first_ten = []
            color_idx = 0
            for traj in all_paths:
                succ_lanelet_trajectory = [(p.x, p.y) for p in traj]
                succ_lanelet_trajectory = np.array(succ_lanelet_trajectory)[4::5, :]
                trajectory_dict[second]["trajectories"].append(succ_lanelet_trajectory)

                if len(succ_lanelet_trajectory) >= 10:
                    succ_traj_first_ten.append(succ_lanelet_trajectory[:10])
                ax.scatter(succ_lanelet_trajectory[:, 0], succ_lanelet_trajectory[:, 1], c=traj_color[color_idx])
                color_idx += 1
            
            succ_traj_first_ten = np.array(succ_traj_first_ten)
            print('succ_traj_first_ten', succ_traj_first_ten.shape)
            predecessor = router.previous(best_lanelet)[0]
            threshold = 100.0
            # Get successors of the predecessor
            if succ_traj_first_ten.shape[0] > 0:
                for pred_succ in router.following(predecessor):
                    proj_pt3d = project(pred_succ.centerline, proj_pt)
                    s0 = compute_s0(pred_succ.centerline, proj_pt3d)

                    if path_length(pred_succ.centerline) - s0 > 5.0:
                        pred_succ_lanelet_trajectory = [(p.x, p.y) for p in pred_succ.centerline]
                        pred_succ_lanelet_trajectory = np.array(pred_succ_lanelet_trajectory)[4::5, :]
                        if pred_succ_lanelet_trajectory.shape[0] > 10:
                            dist = np.sum((succ_traj_first_ten - pred_succ_lanelet_trajectory[:10, :]) ** 2, axis=(1,2))

                            if all(distance > 100 for distance in dist.tolist()):
                                print("doing predecessor")
                                ax.scatter(pred_succ_lanelet_trajectory[:, 0], pred_succ_lanelet_trajectory[:, 1], c="green")
                                trajectory_dict[second]["trajectories"].append(pred_succ_lanelet_trajectory)

            
            all_paths = collect_trajectories_v3(
                    lanelet_map,
                    router,
                    best_lanelet,
                    pt3d,
                    budget=50.0,
                    connection_tol=15.0,
                    enter_threshold=2.0,
                )

            trajectory_dict[second] = {}
            trajectory_dict[second]["trajectories"] = []
            for traj in all_paths:
                sliced_trajectory = [(p.x, p.y) for p in traj]
                trajectory_dict[second]["trajectories"].append(sliced_trajectory)
                        
            lanelet_trajectory = [(p.x, p.y) for p in best_lanelet.centerline]
            lanelet_trajectory = np.array(lanelet_trajectory)
            '''

            current_lanelets = find_current_lanelets(lanelet_map, vehicle.get_location().x, -vehicle.get_location().y)
            graph = build_routing_graph(lanelet_map)
            trajectory_dict[second] = {}
            trajectory_dict[second]["trajectories"] = []
            trajectory_dict[second]["vehicle_location"] = [vehicle.get_location().x, -vehicle.get_location().y]
            ego_yaw = math.radians(vehicle.get_transform().rotation.yaw)
            raw_paths = []
            for lanelet in current_lanelets:
                paths = get_candidate_paths(lanelet_map, graph, lanelet, max_distance=50.0)
                for path in paths:
                    traj = lanelet_sequence_to_trajectory(path, step=0.5)
                    raw_paths.append(traj)
                    

            ego_yaw = math.radians(-vehicle.get_transform().rotation.yaw)
            filtered_paths = filter_trajectories_by_initial_direction(raw_paths, ego_yaw, max_angle_deg=60.0)
            if not filtered_paths:
                # you just turned — allow up to 120° until you’re fully on the new lane
                filtered_paths = filter_trajectories_by_initial_direction(raw_paths, ego_yaw, max_angle_deg=120.0)

            for traj in filtered_paths:
                possible_trajectory = [(p[0], p[1]) for p in traj]
                trajectory_dict[second]["trajectories"].append(possible_trajectory)

            if second == 100:
                break


            second+=1
            
#--------------------------------------------------------------------------------------------------------------------
        
        

except KeyboardInterrupt:
    print("User interrupted the script.")
finally:
    vehicle.set_autopilot(False)
    settings.fixed_delta_seconds = 0
    settings.synchronous_mode = False
    world.apply_settings(settings)

    for vehicle in world.get_actors().filter('*vehicle*'):
        vehicle.destroy()

second: 0
second: 1
second: 2
second: 3
second: 4
second: 5
second: 6
second: 7
second: 8
second: 9
second: 10
second: 11
second: 12
second: 13
second: 14
second: 15
second: 16
second: 17
second: 18
second: 19
second: 20
second: 21
second: 22
second: 23
second: 24
second: 25
second: 26
second: 27
second: 28
second: 29
second: 30
second: 31
second: 32
second: 33
second: 34
second: 35
second: 36
second: 37
second: 38
second: 39
second: 40
second: 41
second: 42
second: 43
second: 44
second: 45
second: 46
second: 47
second: 48
second: 49
second: 50
second: 51
second: 52
second: 53
second: 54
second: 55
second: 56
second: 57
second: 58
second: 59
second: 60
second: 61
second: 62
second: 63
second: 64
second: 65
second: 66
second: 67
second: 68
second: 69
second: 70
second: 71
second: 72
second: 73
second: 74
second: 75
second: 76
second: 77
second: 78
second: 79
second: 80
second: 81
second: 82
second: 83
second: 84
second: 85
second: 86
second: 87
second: 88
second: 89
second: 90
second: 9

In [7]:
with open("./road_boundary_coord.json", "r") as file:
    road_boundary_coord = json.load(file)

with open("./road_center_coord.json", "r") as file:
    road_center_coord = json.load(file)

road_center_coord = np.array(road_center_coord)
road_boundary_coord = np.array(road_boundary_coord)

#### plotting to visualize trajectories captured in loop

In [ ]:
print(len(trajectory_dict.keys()))

for i in range(100):
    #plt.figure(figsize=(30, 30))
    fig, ax = plt.subplots(figsize=(30,30))
    ax.scatter(road_boundary_coord[:, 0], road_boundary_coord[:, 1], c="gray")
    ax.scatter(trajectory_dict[i]["vehicle_location"][0], trajectory_dict[i]["vehicle_location"][1], c="blue", s=100)

    ego_xy = (trajectory_dict[i]["vehicle_location"][0], trajectory_dict[i]["vehicle_location"][1])
    plot_color = ["red", "blue", "green"]
    color_idx = 0
    for trajectory in trajectory_dict[i]["trajectories"]:
        trimmed = slice_trajectory_ahead_vec(trajectory, ego_xy)
        trimmed = np.array(trimmed)
        ax.scatter(trimmed[4::5, 0], trimmed[4::5, 1], c=plot_color[color_idx])
        color_idx += 1
    plt.title(len(trajectory_dict[i]["trajectories"]))
    

    plt.savefig(f"./plot/{i}.png")
    plt.close()


: 

#### testing possible trajectory slice function

In [ ]:
second = 6
path_idx = 0

ego_xy = (trajectory_dict[second]["vehicle_location"][0], trajectory_dict[second]["vehicle_location"][1])
trimmed = slice_trajectory_ahead_vec(trajectory_dict[second]["trajectories"][path_idx].tolist(), ego_xy)
trimmed = np.array(trimmed)
print(trimmed.shape)

fig, ax = plt.subplots(figsize=(30,30))
ax.scatter(road_boundary_coord[:, 0], road_boundary_coord[:, 1], c="gray")
ax.scatter(trajectory_dict[second]["vehicle_location"][0], trajectory_dict[second]["vehicle_location"][1], c="blue", s=100)
ax.scatter(trimmed[4::5, 0], trimmed[4::5, 1], c="red")

In [3]:

vehicle_bp = bp_lib.find('vehicle.tesla.model3') #112
specified_spawn_position = carla.Transform(carla.Location(x=61.6, y=66.4, z=3.0), carla.Rotation(pitch=-0.0, yaw=-180.0, roll=0.0))

vehicle = world.spawn_actor(vehicle_bp, specified_spawn_position) # spawn_points[136]

In [9]:
vehicle.destroy()

True

#### functions to extract possible trajectories

In [66]:
def find_current_lanelets(lanelet_map, pose_x, pose_y, search_radius=10.0):
    """
    Find lanelets near the vehicle pose and return those containing the point.
    """
    pt2d = BasicPoint2d(pose_x, pose_y)
    # Find nearest lanelets (returns list of (distance, lanelet) pairs)
    nearest = lanelet2.geometry.findNearest(lanelet_map.laneletLayer, pt2d, 10)
    current = []
    for _, lanelet in nearest:
        # Skip non-road lanelets (e.g., crosswalks)
        if "subtype" in lanelet.attributes:
            val = lanelet.attributes["subtype"]
            if val in ["Crosswalk", "Walkway"]:
                continue
        # Check if point is inside the lanelet polygon
        if lanelet2.geometry.inside(lanelet, pt2d):
            current.append(lanelet)
    return current

def build_routing_graph(lanelet_map):
    """
    Build a routing graph for vehicular traffic on the map.
    """
    # Use the Lanelet2 traffic rules for vehicles (right-hand traffic assumed)
    #traffic_rules = TrafficRulesFactory.create(lanelet_map, 
    #                TrafficRulesFactory.Type.VEHICLE, lanelet2.traffic_rules.Locations.Germany)
    # Use default routing cost (distance)
    traffic_rules = lanelet2.traffic_rules.create(
        lanelet2.traffic_rules.Locations.Germany,
        lanelet2.traffic_rules.Participants.Vehicle
    )

    graph = RoutingGraph(lanelet_map, traffic_rules)
    return graph

def get_candidate_paths(lanelet_map, routing_graph, start_lanelet, max_distance):
    """
    Enumerate possible lanelet paths up to max_distance ahead of start_lanelet.
    Uses BFS on the routing graph.
    """
    paths = [[start_lanelet]]
    results = []
    # Helper to compute length of a lanelet (arc length)
    def lanelet_length(llet):
        pts = llet.centerline
        length = 0.0
        for i in range(1, len(pts)):
            dx = pts[i].x - pts[i-1].x
            dy = pts[i].y - pts[i-1].y
            length += (dx*dx + dy*dy)**0.5
        return length

    # BFS expansion of paths
    from collections import deque
    queue = deque(paths)
    while queue:
        path = queue.popleft()
        last = path[-1]
        # Compute path length so far
        length_so_far = sum(lanelet_length(l) for l in path)
        if length_so_far > max_distance:
            # path is long enough
            results.append(path)
            continue
        # Extend path by successors
        successors = routing_graph.following(last, withLaneChanges=True)
        for next_lanelet in successors:
            # Avoid loops
            if next_lanelet in path:
                continue
            # Filter out non-road lanelets
            if "subtype" in next_lanelet.attributes:
                val = next_lanelet.attributes["subtype"]
                if val in ["Crosswalk", "Walkway"]:
                    continue
            new_path = path + [next_lanelet]
            queue.append(new_path)
        # If no successors or all filtered, finalize this branch
        if not successors:
            results.append(path)
    return results

def lanelet_sequence_to_trajectory(path, step=1.0):
    """
    Convert a sequence of lanelets into an interpolated trajectory (list of 3D points).
    """
    traj = []
    # Concatenate centerline points
    for i, lanelet in enumerate(path):
        line = lanelet.centerline
        # For intermediate lanelets, skip first point to avoid duplicates
        points = list(line) #.basicLineString()  # gets list of points
        if i > 0 and traj:
            points = points[1:]
        for p in points:
            traj.append((p.x, p.y, p.z))
    # Simple linear interpolation to uniform spacing (could use spline)
    interpolated = []
    if not traj:
        return interpolated
    accumulated = 0.0
    interpolated.append(traj[0])
    for i in range(1, len(traj)):
        (x0,y0,z0) = traj[i-1]
        (x1,y1,z1) = traj[i]
        dx = x1 - x0; dy = y1 - y0; dz = z1 - z0
        segment_len = (dx*dx + dy*dy + dz*dz)**0.5
        steps = max(int(segment_len/step), 1)
        for j in range(1, steps+1):
            t = j/steps
            x = x0 + dx*t; y = y0 + dy*t; z = z0 + dz*t
            interpolated.append((x,y,z))
    return interpolated

def filter_trajectories_by_initial_direction(trajectories, ego_yaw, max_angle_deg=60.0): #  max_angle_deg=60.0
    def angle_diff(a, b):
        d = (a - b + math.pi) % (2*math.pi) - math.pi
        return abs(d)
    thresh = math.radians(max_angle_deg)
    out = []
    for traj in trajectories:
        if len(traj) < 2:
            continue
        dx = traj[1][0] - traj[0][0]
        dy = traj[1][1] - traj[0][1]
        traj_yaw = math.atan2(dy, dx)
        if angle_diff(traj_yaw, ego_yaw) <= thresh:
            out.append(traj)
    return out

def slice_trajectory_ahead(traj_xy, ego_xy):
    """
    Given
      - traj_xy: list of (x,y) points forming a continuous polyline
      - ego_xy:  (x,y) of your vehicle in the same frame
    Returns
      - new_traj: list starting at the projection of ego_xy onto traj_xy,
                  followed by all original points that lie ahead.
    """
    ego_x, ego_y = ego_xy

    # Precompute segment lengths & cumulative distances
    seg_lengths = []
    for i in range(len(traj_xy)-1):
        dx = traj_xy[i+1][0] - traj_xy[i][0]
        dy = traj_xy[i+1][1] - traj_xy[i][1]
        seg_lengths.append(math.hypot(dx, dy))
    cum_dist = [0.0]
    for L in seg_lengths:
        cum_dist.append(cum_dist[-1] + L)

    # Find the closest projection on any segment
    best_i, best_u, best_d = 0, 0.0, float('inf')
    for i in range(len(seg_lengths)):
        x0,y0 = traj_xy[i]
        x1,y1 = traj_xy[i+1]
        dx, dy = x1-x0, y1-y0
        ux, uy = ego_x-x0, ego_y-y0
        denom = dx*dx + dy*dy
        if denom <= 0:
            continue
        u = (ux*dx + uy*dy) / denom
        u_clamped = max(0.0, min(1.0, u))
        cx = x0 + u_clamped*dx
        cy = y0 + u_clamped*dy
        d = math.hypot(ego_x-cx, ego_y-cy)
        if d < best_d:
            best_d, best_i, best_u = d, i, u_clamped

    # Compute the projected start point
    x0,y0 = traj_xy[best_i]
    x1,y1 = traj_xy[best_i+1]
    proj_x = x0 + best_u*(x1-x0)
    proj_y = y0 + best_u*(y1-y0)

    # Build the new trimmed trajectory
    new_traj = [(proj_x, proj_y)]
    # Append all subsequent original vertices
    for j in range(best_i+1, len(traj_xy)):
        new_traj.append(traj_xy[j])

    return new_traj

def slice_trajectory_ahead_vec(traj_xy, ego_xy):
    """
    Vectorized version of slice_trajectory_ahead.
    traj_xy: (N,2) numpy array of (x,y)
    ego_xy:  (2,) tuple or array
    Returns trimmed trajectory as a (M,2) array.
    """
    pts   = np.asarray(traj_xy, dtype=float)
    ego   = np.asarray(ego_xy,   dtype=float)

    # compute segment vectors and lengths
    vecs  = pts[1:] - pts[:-1]              # shape (N-1,2)
    seg2  = np.sum(vecs**2, axis=1)         # squared norms, shape (N-1,)

    # vector from each segment start to ego
    rel   = ego - pts[:-1]                  # shape (N-1,2)

    # projection parameter u (unclamped), shape (N-1,)
    u_raw = np.sum(rel * vecs, axis=1) / seg2
    u     = np.clip(u_raw, 0.0, 1.0)

    # projection points
    proj = pts[:-1] + (vecs.T * u).T        # shape (N-1,2)

    # distances from ego to each proj
    d2   = np.sum((proj - ego)**2, axis=1)  # squared distances
    idx  = np.argmin(d2)                    # best segment index

    # build the trimmed array
    best_u = u[idx]
    p0, p1 = pts[idx], pts[idx+1]
    proj_pt = p0 + best_u * (p1 - p0)       # single projection

    # stack proj_pt and all subsequent pts
    trimmed = np.vstack([proj_pt, pts[idx+1:]])

    return trimmed

#### How to extract possible trajectories inspired by code from map_based_prediction

In [ ]:
# Example usage:
# lanelet_map = ...  # Loaded LaneletMap with lanelets
# graph = build_routing_graph(lanelet_map)
# obj_pose = ...     # Vehicle pose (with x, y fields)
# current_lanelets = find_current_lanelets(lanelet_map, obj_pose)
# candidate_trajectories = []
# for lanelet in current_lanelets:
#     paths = get_candidate_paths(lanelet_map, graph, lanelet, max_distance=50.0)
#     for path in paths:
#         traj = lanelet_sequence_to_trajectory(path, step=0.5)
#         candidate_trajectories.append(traj)


map_path = os.path.expanduser('~/Documents/town_10/backup/lanelet2_map.osm')
assert os.path.exists(map_path), f"Map file not found at {map_path}"
proj = LocalCartesianProjector(Origin(0, 0, 0))
lanelet_map = load(map_path, proj)

current_lanelets = find_current_lanelets(lanelet_map, vehicle.get_location().x, -vehicle.get_location().y)


#lanelet_trajectory = [(p.x, p.y) for p in current_lanelets[0].centerline]
#lanelet_trajectory = np.array(lanelet_trajectory)[4::5, :]
plt.figure(figsize=(30, 30))
plt.scatter(road_boundary_coord[:, 0], road_boundary_coord[:, 1], c="gray")
plt.scatter(vehicle.get_location().x, -vehicle.get_location().y, c="blue", s=100)
#plt.scatter(lanelet_trajectory[:, 0], lanelet_trajectory[:, 1], c="blue")


graph = build_routing_graph(lanelet_map)
candidate_trajectories = []
for lanelet in current_lanelets:
    paths = get_candidate_paths(lanelet_map, graph, lanelet, max_distance=50.0)
    for path in paths:
        traj = lanelet_sequence_to_trajectory(path, step=0.5)
        candidate_trajectories.append(traj)
        print(traj)
        possible_trajectory = [(p[0], p[1]) for p in traj]
        possible_trajectory = np.array(possible_trajectory)
        plt.scatter(possible_trajectory[2::3,0], possible_trajectory[2::3,1], c="green")
        print(possible_trajectory.shape)

ego_yaw = math.radians(vehicle.get_transform().rotation.yaw)
raw_paths = candidate_trajectories   # from your lanelet_sequence_to_trajectory()
filtered_paths = filter_trajectories_by_initial_direction(raw_paths, ego_yaw, max_angle_deg=60.0)


plt.show()

